# MiniGPT, running: follow along

This workbook goes with my post [MiniGPT (Part 1)](https://haddley.github.io/posts/minigpt/). It runs my trained MiniGPT model, the *exhibit*, and reproduces every number in the post, step by step, from the letter IDs to the wheel of chances. It only *runs* the model; how the model was trained is the subject of [Part 2](https://haddley.github.io/posts/minigpt-grown/).

Run the cells from top to bottom. A CPU is enough: nothing here needs a GPU.

The model code in section 1 is copied from Jibin Joseph's [MiniGPT notebook](https://github.com/jibin10/MiniGPT), sections 1.1 to 1.5, under its MIT licence (notice at the end).

## 1. The model code

These cells are the notebook's own classes, unchanged: the settings (`GPTConfig`), attention (`CausalSelfAttention`), the MLP (`FeedForward`), one block (`TransformerBlock`), and the whole model (`MiniGPT`). They define the machine, but contain no trained numbers.

In [1]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
@dataclass
class GPTConfig:
    # Maximum number of tokens the model can look at at once.
    block_size: int = 128
    # Number of tokens in the vocabulary - for Part 1 testing.
    vocab_size: int = 65
    # Number of Transformer blocks stacked in the model.
    n_layer: int = 4
    # Number of attention heads inside each self-attention module.
    n_head: int = 4
    # Size of each token embedding vector.
    n_embd: int = 128
    # Dropout probability used for regularization.
    dropout: float = 0.1

In [3]:
# Causal multi-head self-attention module.
class CausalSelfAttention(nn.Module):
    # Initialize the attention module.
    def __init__(self, config):
        super().__init__()
        # Check that the embedding dimension can be evenly split across attention heads.
        assert config.n_embd % config.n_head == 0, "n_embd must be divisible by n_head"

        # Store the number of attention heads.
        self.n_head = config.n_head
        # Store the embedding dimension.
        self.n_embd = config.n_embd
        # Compute the size of each attention head.
        self.head_dim = config.n_embd // config.n_head

        # Create the query, key, value projection layers.
        self.query = nn.Linear(config.n_embd, config.n_embd)
        self.key = nn.Linear(config.n_embd, config.n_embd)
        self.value = nn.Linear(config.n_embd, config.n_embd)

        # Create the output projection layer after attention heads are combined.
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        # Create dropout for attention probabilities and after the output projection.
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # Create a lower-triangular causal mask of shape (block_size, block_size).
        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size, dtype=torch.bool))
        # Reshape the mask to broadcast over batch and heads: (1, 1, block_size, block_size).
        causal_mask = causal_mask.view(1, 1, config.block_size, config.block_size)

        # Register the mask as a buffer so it moves with the model but is not a trainable parameter.
        self.register_buffer("causal_mask", causal_mask)

    # Define the forward pass.
    def forward(self, x):
        # Read batch size B, sequence length T, and embedding size C from input.
        B, T, C = x.shape
        # Project input embeddings into query, key, value vectors.
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Reshape queries, keys, values from (B, T, C) to (B, T, n_head, head_dim).
        q = q.view(B, T, self.n_head, self.head_dim)
        k = k.view(B, T, self.n_head, self.head_dim)
        v = v.view(B, T, self.n_head, self.head_dim)

        # Move the head dimension before the time dimension: (B, n_head, T, head_dim).
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Compute raw attention scores using query-key dot products.
        scores = q @ k.transpose(-2, -1)

        # Scale attention scores by sqrt(head_dim) for stable softmax values.
        scores = scores / math.sqrt(self.head_dim)
        # Select only the part of the causal mask needed for the current sequence length T.
        mask = self.causal_mask[:, :, :T, :T]

        # Put -infinity where the model should not attend, especially future positions.
        scores = scores.masked_fill(mask == False, float("-inf"))

        # Convert attention scores into attention probabilities.
        attn = F.softmax(scores, dim=-1)
        # Apply dropout to attention probabilities.
        attn = self.attn_dropout(attn)
        # Multiply attention probabilities by value vectors to get context vectors.
        out = attn @ v

        # Move the time dimension back before the head dimension: (B, T, n_head, head_dim).
        out = out.transpose(1, 2)
        # Make tensor memory layout contiguous before reshaping.
        out = out.contiguous()
        # Merge all attention heads back into the embedding dimension: (B, T, C).
        out = out.view(B, T, C)
        # Apply final linear projection after combining heads.
        out = self.proj(out)
        # Apply residual dropout after projection.
        out = self.resid_dropout(out)

        # Return the attention output.
        return out

In [4]:
# Feed-forward MLP used inside each Transformer block.
class FeedForward(nn.Module):
    # Initialize the MLP.
    def __init__(self, config):
        super().__init__()

        # First linear layer expands the embedding dimension by 4x.
        self.fc1 = nn.Linear(config.n_embd, 4 * config.n_embd)
        # GELU activation is commonly used in GPT-style Transformers.
        self.gelu = nn.GELU()
        # Second linear layer projects the hidden dimension back to n_embd.
        self.fc2 = nn.Linear(4 * config.n_embd, config.n_embd)
        # Dropout is used for regularization.
        self.dropout = nn.Dropout(config.dropout)

    # Define the forward pass for the MLP.
    def forward(self, x):
        # Apply the first linear layer.
        x = self.fc1(x)
        # GELU nonlinearity.
        x = self.gelu(x)
        # Second linear layer.
        x = self.fc2(x)
        # Dropout.
        x = self.dropout(x)

        # Return the MLP output.
        return x

In [5]:
# Decoder-only Transformer block.
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        # LayerNorm is applied before self-attention.
        self.ln1 = nn.LayerNorm(config.n_embd)

        # Causal self-attention module.
        self.attn = CausalSelfAttention(config)

        # Second LayerNorm is applied before the MLP.
        self.ln2 = nn.LayerNorm(config.n_embd)

        # Feed-forward MLP module.
        self.mlp = FeedForward(config)

    # Forward pass for one Transformer block.
    def forward(self, x):
        # LayerNorm, then attention, then add the residual connection.
        x = x + self.attn(self.ln1(x))

        # LayerNorm, then MLP, then add the residual connection.
        x = x + self.mlp(self.ln2(x))

        # Updated hidden states.
        return x

In [6]:
class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        # Token embedding table maps token IDs to vectors.
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)

        # Positional embedding table maps token positions to vectors.
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)

        # Dropout is applied after adding token and position embeddings.
        self.dropout = nn.Dropout(config.dropout)

        # List of Transformer blocks.
        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layer)
        ])

        # Final LayerNorm before the language-model head.
        self.final_ln = nn.LayerNorm(config.n_embd)

        # Linear language-model head maps hidden states to vocabulary logits.
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)

        # Custom weight initialization to all suitable layers.
        self.apply(self._init_weights)

    # Weight initialization function.
    def _init_weights(self, module):
        # Check whether the module is a Linear layer.
        if isinstance(module, nn.Linear):
            # Initialize Linear layer weights from a normal distribution.
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            # Check whether the Linear layer has a bias term.
            if module.bias is not None:
                # Initialize Linear layer bias values to zero.
                nn.init.zeros_(module.bias)

        # Check whether the module is an Embedding layer.
        elif isinstance(module, nn.Embedding):
            # Initialize Embedding weights from a normal distribution.
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # Count the total number of trainable model parameters.
    def count_parameters(self):
        # Sum the number of elements in every trainable parameter tensor.
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    # Forward pass of the GPT model.
    def forward(self, idx, targets=None):
        # Read batch size B and sequence length T from token ID tensor.
        B, T = idx.shape

        # Make sure the input sequence is not longer than the model context length.
        assert T <= self.config.block_size, "Input sequence is longer than block_size"

        # Create position IDs from 0 to T-1 on the same device as the input.
        pos = torch.arange(0, T, device=idx.device)

        # Convert token IDs into token embeddings of shape (B, T, n_embd).
        tok_emb = self.token_embedding(idx)

        # Convert position IDs into position embeddings of shape (T, n_embd).
        pos_emb = self.position_embedding(pos)

        # Add token embeddings and position embeddings using broadcasting.
        x = tok_emb + pos_emb

        # Dropout after the embedding sum.
        x = self.dropout(x)

        # Pass the hidden states through each Transformer block.
        for block in self.blocks:
            # Update hidden states using one Transformer block.
            x = block(x)

        # Final LayerNorm.
        x = self.final_ln(x)

        # Convert final hidden states into vocabulary logits.
        logits = self.lm_head(x)

        # Start with no loss by default.
        loss = None

        # Compute loss only when target tokens are provided.
        if targets is not None:
            # Flatten logits from (B, T, vocab_size) to (B*T, vocab_size).
            logits_flat = logits.view(B * T, self.config.vocab_size)

            # Flatten targets from (B, T) to (B*T).
            targets_flat = targets.view(B * T)

            # Compute cross-entropy loss for next-token prediction.
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

## 2. Load my trained numbers

The checkpoint `exhibit.pt` holds all 826,433 fixed numbers. `chars` is the vocabulary: the 65 letters of Tiny Shakespeare, sorted, so that each letter's ID is its place in the list.

In [7]:
import urllib.request
urllib.request.urlretrieve("https://haddley.github.io/minigpt-demo/exhibit.pt", "exhibit.pt")

chars = list("\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
stoi = {ch: i for i, ch in enumerate(chars)}

model = MiniGPT(GPTConfig())
model.load_state_dict(torch.load("exhibit.pt", map_location="cpu"))
model.eval()                 # running, not training: dropout off
model.requires_grad_(False)  # and no records for training

print(sum(p.numel() for p in model.parameters()), "fixed numbers loaded")

826433 fixed numbers loaded


A small helper: the chances for the next letter after any text, and a way to print the top few.

In [8]:
def chances(text, temperature=1.0):
    idx = torch.tensor([[stoi[ch] for ch in text]])
    logits, _ = model(idx[:, -model.config.block_size:])
    return torch.softmax(logits[0, -1] / temperature, dim=-1)

def show_top(p, n=5):
    for prob, i in zip(*torch.topk(p, n)):
        print(f"{chars[i]!r:6} {prob.item():6.1%}")

## 3. The guessing game

The opening example, exactly as the live demo starts: the speaker's name, a new line, and the line with its last letter hidden.

In [9]:
show_top(chances("KING RICHARD III:\nA horse! a horse! my kingdom for a hors"), 3)

'e'     76.6%
','      4.6%
'.'      3.9%


After `go`, the chances are spread out. One more letter, and they are not:

In [10]:
show_top(chances("go"), 4)
print()
show_top(chances("goo"), 4)

'o'     34.8%
' '     15.6%
'd'     15.4%
'n'      5.7%

'd'     96.6%
'k'      0.6%
'r'      0.6%
's'      0.4%


## 4. Step 1: letters to numbers

In [11]:
ids = [stoi[ch] for ch in "goo"]
ids

[45, 53, 53]

## 5. Step 2: letter cards, position cards, and working cards

The letter cards and position cards are fixed tables. Adding a letter card to its position card makes a working card, one per position.

In [12]:
g_card   = model.token_embedding.weight[stoi["g"]]   # the g letter card
pos1     = model.position_embedding.weight[0]         # the position 1 card (Python counts from 0)
working1 = g_card + pos1                              # working card 1, before block 1
print("g letter card:    ", g_card[:4])
print("position 1 card:  ", pos1[:4])
print("working card 1:   ", working1[:4])

g letter card:     tensor([-0.0492,  0.0315,  0.0139,  0.0300])
position 1 card:   tensor([ 0.0430, -0.0322,  0.0407,  0.0705])
working card 1:    tensor([-0.0062, -0.0007,  0.0546,  0.1005])


All three working cards for `goo`, exactly as `MiniGPT.forward` makes them:

In [13]:
idx = torch.tensor([ids])
x = model.token_embedding(idx) + model.position_embedding(torch.arange(3))
print(x.shape)   # 1 text, 3 working cards, 128 numbers each

torch.Size([1, 3, 128])


Neighbouring position cards end up alike, and far-apart ones point the opposite way (the post's position ruler):

In [14]:
P = F.normalize(model.position_embedding.weight, dim=1)
sim = P @ P.T
print("neighbours:     ", round(sim.diagonal(1).mean().item(), 2))
print("100 apart:      ", round(sim.diagonal(100).mean().item(), 2))

neighbours:      0.65
100 apart:       -0.35


## 6. Step 3, inside block 1: attention

Normalise the working cards, then make the query, key, and value cards with block 1's three fixed recipes.

In [15]:
block1 = model.blocks[0]
xn = block1.ln1(x)                      # normalised working cards
q = block1.attn.query(xn)[0]            # query cards: 3 x 128
k = block1.attn.key(xn)[0]
v = block1.attn.value(xn)[0]
print("working card 3, normalised:", xn[0, 2, :3])
print("its query card, head 1:   ", q[2, :3])

working card 3, normalised: tensor([ 0.4256, -2.0958,  0.2216])
its query card, head 1:    tensor([-0.0048,  0.2656,  0.7558])


Every number on a query card is a weighted mix of all 128 numbers on the working card, plus a bias. Here is the second number of working card 3's query card, worked out by hand:

In [16]:
W, b = block1.attn.query.weight, block1.attn.query.bias
terms = W[1] * xn[0, 2]                 # 128 weight x number products
print("first three terms:", [round(t, 3) for t in terms[:3].tolist()])
print("sum of 128 terms + bias:", round((terms.sum() + b[1]).item(), 2))

first three terms: [-0.015, 0.079, 0.007]
sum of 128 terms + bias: 0.27


Head 1 uses the first 32 numbers of each card. Match working card 3's query against every key, shrink by √32, and share out with softmax:

In [17]:
hd = 32
scores = q[2, :hd] @ k[:, :hd].T        # query x key, added up
shrunk = scores / math.sqrt(hd)
shares = torch.softmax(shrunk, dim=-1)
for name, row in [("query x key", scores), ("divided by sqrt(32)", shrunk), ("share of attention", shares)]:
    print(f"{name:20}", [round(t, 2) for t in row.tolist()])

query x key          [-2.19, 15.6, -3.62]
divided by sqrt(32)  [-0.39, 2.76, -0.64]
share of attention   [0.04, 0.93, 0.03]


All four heads of block 1, for working card 3 (each head uses its own 32-number piece):

In [18]:
for h in range(4):
    s = slice(h * hd, (h + 1) * hd)
    sh = torch.softmax(q[2, s] @ k[:, s].T / math.sqrt(hd), dim=-1)
    print(f"head {h + 1}:", [f"{t:.1%}" for t in sh.tolist()])

head 1: ['4.0%', '92.9%', '3.1%']
head 2: ['83.4%', '12.2%', '4.4%']
head 3: ['83.4%', '14.9%', '1.6%']
head 4: ['24.7%', '66.5%', '8.8%']


## 7. Four blocks in a row

Each block adds to the working cards: working card out = working card in + what attention adds + what the MLP adds. Here is what each block does to working card 3.

In [19]:
x0 = x.clone()
x_run = x.clone()
for n, blk in enumerate(model.blocks, start=1):
    a = blk.attn(blk.ln1(x_run)); y = x_run + a
    d = blk.mlp(blk.ln2(y));      z = y + d
    print(f"block {n}: size in {x_run[0,2].norm():.2f}, attention adds {a[0,2].norm():.2f}, "
          f"MLP adds {d[0,2].norm():.2f}, size out {z[0,2].norm():.2f}, "
          f"in-vs-out likeness {F.cosine_similarity(x_run[0,2], z[0,2], dim=0):.2f}")
    x_run = z
print("likeness to the input embedding after block 4:", round(F.cosine_similarity(x0[0,2], x_run[0,2], dim=0).item(), 2))

block 1: size in 0.50, attention adds 0.59, MLP adds 1.66, size out 2.16, in-vs-out likeness 0.47
block 2: size in 2.16, attention adds 0.80, MLP adds 0.85, size out 2.72, in-vs-out likeness 0.90
block 3: size in 2.72, attention adds 0.90, MLP adds 1.16, size out 2.92, in-vs-out likeness 0.90
block 4: size in 2.92, attention adds 0.74, MLP adds 1.86, size out 3.09, in-vs-out likeness 0.78
likeness to the input embedding after block 4: 0.08


Stopping early (the *logit lens*): send the last working card to the answer cards after each block.

In [20]:
text = "First Citizen:\nBefore we proceed any further, hear me spea"
idx2 = torch.tensor([[stoi[ch] for ch in text]])
h = model.token_embedding(idx2) + model.position_embedding(torch.arange(idx2.shape[1]))
def peek(h):
    p = torch.softmax(model.lm_head(model.final_ln(h[0, -1])), dim=-1)
    return ", ".join(f"{chars[i]} {p[i]:.0%}" for i in torch.topk(p, 3).indices)
print("before block 1:", peek(h))
for n, blk in enumerate(model.blocks, start=1):
    h = blk(h)
    print(f"after block {n}: ", peek(h))

before block 1: y 12%, l 11%, d 6%
after block 1:  t 25%, i 17%, c 17%
after block 2:  c 34%, t 21%, r 11%
after block 3:  k 66%, r 13%, c 5%
after block 4:  k 98%, r 1%, d 0%


## 8. Step 4: the answer cards

Normalise the last working card once more, then score it against all 65 answer cards: multiply, add up, add the bias.

In [21]:
last = model.final_ln(x_run[0, 2])
scores = model.lm_head.weight @ last + model.lm_head.bias
for ch in "dkrs":
    print(repr(ch), "score", round(scores[stoi[ch]].item(), 2))
print("lowest:", repr(chars[scores.argmin()]), round(scores.min().item(), 2))
p = torch.softmax(scores, dim=-1)
print("d chance:", f"{p[stoi['d']]:.1%}")

'd' score 8.58
'k' score 3.45
'r' score 3.41
's' score 3.01
lowest: 'M' -5.77
d chance: 96.6%


Is the guess just the closest letter card? Not in this model: compare the last working card with the letter cards and with the answer cards.

In [22]:
def rank_of_d(table):
    sim = F.cosine_similarity(last[None], table, dim=1)
    return (sim.argsort(descending=True) == stoi["d"]).nonzero().item() + 1
print("d's rank among the letter cards:", rank_of_d(model.token_embedding.weight))
print("d's rank among the answer cards:", rank_of_d(model.lm_head.weight))

d's rank among the letter cards: 47
d's rank among the answer cards: 1


## 9. Step 5: temperature, trimming, and spinning the wheel

In [23]:
for t in (0.5, 1.0, 2.0):
    p = chances("good m", temperature=t)
    print(f"temperature {t}:", ", ".join(f"{ch} {p[stoi[ch]]:.1%}" for ch in "yea"))

temperature 0.5: y 61.1%, e 26.6%, a 8.5%
temperature 1.0: y 40.8%, e 26.9%, a 15.2%
temperature 2.0: y 25.9%, e 21.0%, a 15.8%


Top-p 90%: keep the biggest slices until they add up to 90%, then stretch them to fill the wheel.

In [24]:
p = chances("good m")
order = p.argsort(descending=True)
kept, total = [], 0.0
for i in order.tolist():
    kept.append(i); total += p[i].item()
    if total >= 0.9: break
print(len(kept), "slices kept:", [chars[i] for i in kept], f"holding {total:.1%}")

5 slices kept: ['y', 'e', 'a', 'o', 'i'] holding 96.7%


Spin the wheel 200 times. The first line fixes the spins, so you get the same text as the post; remove it for a different text each time.

In [25]:
torch.manual_seed(1)
idx = torch.tensor([[stoi[ch] for ch in "ROMEO:\n"]])
for _ in range(200):
    logits, _ = model(idx[:, -model.config.block_size:])
    probs = torch.softmax(logits[0, -1] / 0.8, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    idx = torch.cat([idx, next_id.view(1, 1)], dim=1)
print("".join(chars[i] for i in idx[0].tolist()))

ROMEO:
What suqners the caple that me, by my shall with a virtue,
And here, in with poor all gainst herse mage, which your do
By countemptisanion at Erevols, he braved-hame to you will in their follow.

LORD


## 10. Where every fixed number lives

In [26]:
for name, t in model.state_dict().items():
    if name.startswith("blocks.") and not name.startswith("blocks.0."):
        continue
    if "causal_mask" in name:
        continue
    print(f"{name:32} {str(tuple(t.shape)):12} {t.numel():>7,}")
print("total:", f"{sum(p.numel() for p in model.parameters()):,}")

token_embedding.weight           (65, 128)      8,320
position_embedding.weight        (128, 128)    16,384
blocks.0.ln1.weight              (128,)           128
blocks.0.ln1.bias                (128,)           128
blocks.0.attn.query.weight       (128, 128)    16,384
blocks.0.attn.query.bias         (128,)           128
blocks.0.attn.key.weight         (128, 128)    16,384
blocks.0.attn.key.bias           (128,)           128
blocks.0.attn.value.weight       (128, 128)    16,384
blocks.0.attn.value.bias         (128,)           128
blocks.0.attn.proj.weight        (128, 128)    16,384
blocks.0.attn.proj.bias          (128,)           128
blocks.0.ln2.weight              (128,)           128
blocks.0.ln2.bias                (128,)           128
blocks.0.mlp.fc1.weight          (512, 128)    65,536
blocks.0.mlp.fc1.bias            (512,)           512
blocks.0.mlp.fc2.weight          (128, 512)    65,536
blocks.0.mlp.fc2.bias            (128,)           128
final_ln.weight             

## 11. The same model in llama.cpp, via GGUF

[llama.cpp](https://github.com/ggml-org/llama.cpp) runs models stored as a single GGUF file. My [export script](https://haddley.github.io/minigpt-demo/export_gguf.py) maps MiniGPT onto llama.cpp's GPT-2 design. This cell exports the file, then gets llama.cpp: the Homebrew one if you have it (`brew install llama.cpp` on a Mac), or else the official Linux build, which is what Colab uses.

In [27]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "gguf"], check=True)
urllib.request.urlretrieve("https://haddley.github.io/minigpt-demo/export_gguf.py", "export_gguf.py")
result = subprocess.run([sys.executable, "export_gguf.py", "exhibit.pt", "exhibit.gguf"],
                        check=True, capture_output=True, text=True)
print(result.stdout.strip(), "-", os.path.getsize("exhibit.gguf"), "bytes")

wrote exhibit.gguf - 3309984 bytes


In [28]:
import os, shutil, tarfile
llama = shutil.which("llama-completion")
if llama is None:   # Colab and other Linux machines: download the official build
    tag = "b11406"
    urllib.request.urlretrieve(
        f"https://github.com/ggml-org/llama.cpp/releases/download/{tag}/llama-{tag}-bin-ubuntu-x64.tar.gz", "llama.tar.gz")
    tarfile.open("llama.tar.gz").extractall()
    llama = os.path.abspath(f"llama-{tag}/llama-completion")
    os.environ["LD_LIBRARY_PATH"] = os.path.dirname(llama) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print(llama)

/opt/homebrew/bin/llama-completion


In [29]:
out = subprocess.run([llama, "-m", "exhibit.gguf", "-p", "ROMEO:\n", "-n", "120",
                      "--temp", "0.8", "--top-k", "0", "--top-p", "1.0", "--min-p", "0"],
                     capture_output=True, text=True)
print(out.stdout)

ROMEO:
How as mean. Come, right come untertains; do look not in earths
With of a way: this patice be doth my with against me
li




---
### Licence notice for the model code in section 1

MiniGPT, Copyright (c) 2026 Jibin Joseph, https://github.com/jibin10/MiniGPT. The code is licensed under the MIT License:

Permission is hereby granted, free of charge, to any person obtaining a copy of the software and associated documentation files covered by the MIT License (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.